# FlowMetric — hybrid detection, tracking & service events

**~95% local + ~5% GPT‑6 Astra:**
1. YOLO11s @1280 + BoT-SORT на каждом кадре → боксы и покадровые ID
2. GPT‑6 Astra на сэмплах (1 fps, батчи по 16) → ReID + роль staff/customer
3. Синхронизация по IoU: фрагменты треков склеиваются в стабильные ID людей
4. События по траекториям (`events.py`, зоны в `config/camera.json`): вход, выход, кухня, посадка, обслуживание, ожидание
5. Рендер → `data/output/tracked.mp4`, бенчмарк против `benchmark/ground_truth.json` ([Timings.txt](https://github.com/nofriendsatall/video_analytics/blob/main/Timings.txt))

Ключ: `VSELLM_API_KEY` в `.env` ([VseLLM](https://vsellm.ru/models/openai/gpt-6-astra)). Ответ Astra кэшируется в `data/output/astra_reid.json`.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from astra import has_api_key
from metrics import print_report
from track import run_hybrid

ROOT = Path.cwd()  # notebook runs from the project root
VIDEO_IN = next((ROOT / "data").glob("*.mp4"))
OUT_DIR = ROOT / "data" / "output"
ASTRA_CACHE = OUT_DIR / "astra_reid.json"
GT = ROOT / "benchmark" / "ground_truth.json"

USE_ASTRA = has_api_key() or ASTRA_CACHE.exists()
print("input :", VIDEO_IN.name)
print("astra :", "on" if USE_ASTRA else "off (VSELLM_API_KEY not set)")

In [ ]:
hybrid = run_hybrid(
    VIDEO_IN,
    OUT_DIR / "tracked.mp4",
    use_astra=USE_ASTRA,
    astra_cache=ASTRA_CACHE,
    gt=GT,
)

In [ ]:
local = run_hybrid(VIDEO_IN, OUT_DIR / "local" / "tracked.mp4", use_astra=False, gt=GT)

In [ ]:
for name, run in (("local", local), ("hybrid", hybrid)):
    b = run["benchmark"]
    print(
        f"{name:7s} IDs {b['identity']['pred_ids']:3d}  continuity {b['identity']['continuity_mean']:.0%}  "
        f"roles {b['roles']['accuracy']:.0%}  events strict F1 {b['events_strict']['f1']:.2f} "
        f"(R {b['events_strict']['recall']:.2f})  waiting tIoU {b['states']['waiting']['tiou']}"
    )

In [ ]:
cap = cv2.VideoCapture(str(OUT_DIR / "tracked.mp4"))
cap.set(cv2.CAP_PROP_POS_FRAMES, int(11.6 * hybrid["fps"]))
ok, frame = cap.read()
cap.release()
assert ok

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f"[{hybrid['mode']}] local tracks {hybrid['local_tracks']} → final IDs {hybrid['final_ids']}")
plt.axis("off")
plt.show()